In [1]:
!pip install joblib pandas numpy scikit-learn librosa

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import librosa
import os
import joblib

In [2]:
# Định nghĩa hàm ánh xạ valence và arousal sang nhãn cảm xúc
def map_to_emotion(valence, arousal):
    if valence > 5 and arousal > 5:
        return 'HAPPY'
    elif valence < 5 and arousal < 5:
        return 'SAD'
    elif valence > 5 and arousal > 7:
        return 'ENERGY'
    elif valence > 5 and 3 < arousal < 7:
        return 'ROMANTIC'
    elif 3 < valence < 7 and arousal < 3:
        return 'CHILL'
    else:
        return 'OTHER'

In [3]:
# Hàm trích xuất đặc trưng từ file MP3
def extract_features_from_mp3(mp3_path):
    try:
        y, sr = librosa.load(mp3_path, sr=None)
        mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13), axis=1)  # 13 đặc trưng
        chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=sr), axis=1)     # 12 đặc trưng
        spectral_contrast = np.mean(librosa.feature.spectral_contrast(y=y, sr=sr), axis=1)  # 7 đặc trưng
        tempo = librosa.beat.tempo(y=y, sr=sr)[0]                         # 1 đặc trưng
        rms = np.mean(librosa.feature.rms(y=y))                            # 1 đặc trưng
        zcr = np.mean(librosa.feature.zero_crossing_rate(y=y))            # 1 đặc trưng
        features = np.concatenate((mfcc, chroma, spectral_contrast, [tempo, rms, zcr]))
        return features  # Tổng cộng 35 đặc trưng
    except Exception as e:
        print(f"Error processing {mp3_path}: {e}")
        return None

In [4]:
# Chuẩn bị dữ liệu huấn luyện
def prepare_training_data(annotation_path, audio_dir, feature_save_dir="/kaggle/input/music-detection-dataset"):
    # Tạo thư mục lưu đặc trưng nếu chưa tồn tại
    if not os.path.exists(feature_save_dir):
        os.makedirs(feature_save_dir)
    
    # Đường dẫn file lưu đặc trưng
    features_file = os.path.join(feature_save_dir, "features.npy")
    valence_file = os.path.join(feature_save_dir, "valence.npy")
    arousal_file = os.path.join(feature_save_dir, "arousal.npy")

    # Kiểm tra xem các file đặc trưng đã tồn tại chưa
    if (os.path.exists(features_file) and 
        os.path.exists(valence_file) and 
        os.path.exists(arousal_file)):
        print("Tải các đặc trưng đã lưu trước đó...")
        X = np.load(features_file)
        y_valence = np.load(valence_file)
        y_arousal = np.load(arousal_file)
        print(f"Đã tải: {X.shape[0]} mẫu")
        return X, y_valence, y_arousal

    # Nếu chưa có file đặc trưng, tiến hành trích xuất
    annotations = pd.read_csv(annotation_path)
    annotations.columns = annotations.columns.str.strip()

    X = []
    y_valence = []
    y_arousal = []

    for index, row in annotations.iterrows():
        if (index + 1) % 10 == 0:
            print(f"Đang xử lý bài hát {index + 1}/{len(annotations)}: {row['song_id']}")

        song_id = row['song_id']
        mp3_path = os.path.join(audio_dir, f"{int(song_id)}.mp3")
        if os.path.exists(mp3_path):
            features = extract_features_from_mp3(mp3_path)
            if features is not None:
                X.append(features)
                y_valence.append(row['valence_mean'])
                y_arousal.append(row['arousal_mean'])
        else:
            print(f"File not found: {mp3_path}")

    # Chuyển thành numpy array
    X = np.array(X)
    y_valence = np.array(y_valence)
    y_arousal = np.array(y_arousal)

    # Lưu đặc trưng
    print("Lưu đặc trưng đã trích xuất...")
    np.save(features_file, X)
    np.save(valence_file, y_valence)
    np.save(arousal_file, y_arousal)
    
    print(f"Đã lưu: {X.shape[0]} mẫu")
    return X, y_valence, y_arousal

In [5]:
def train_model(X, y_valence, y_arousal):
    # Chia dữ liệu thành tập huấn luyện và kiểm tra
    X_train, X_test, y_valence_train, y_valence_test = train_test_split(X, y_valence, test_size=0.2, random_state=42)
    _, _, y_arousal_train, y_arousal_test = train_test_split(X, y_arousal, test_size=0.2, random_state=42)

    print(f"Kích thước tập huấn luyện: {X_train.shape}, Kích thước tập kiểm tra: {X_test.shape}")

    # Chuẩn hóa đặc trưng
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Định nghĩa lưới tham số cho GridSearchCV
    param_grid = {
        'n_estimators': [50, 100, 200, 300, 500],
        'max_depth': [None, 10, 20, 30, 50, 100],
        'min_samples_split': [2, 5, 10, 50],
        'min_samples_leaf': [1, 2, 4, 10]
    }

    # Khởi tạo mô hình cơ bản
    rf = RandomForestRegressor(random_state=42)

    # GridSearchCV cho valence
    grid_search_valence = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=5,  # 5-fold cross-validation
        scoring='neg_mean_squared_error',
        n_jobs=-1,  # Sử dụng tất cả CPU cores
        verbose=1
    )
    
    # GridSearchCV cho arousal
    grid_search_arousal = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1,
        verbose=1
    )

    # Huấn luyện GridSearchCV
    grid_search_valence.fit(X_train, y_valence_train)
    grid_search_arousal.fit(X_train, y_arousal_train)

    # Lấy các tham số tốt nhất
    best_params_valence = grid_search_valence.best_params_
    best_params_arousal = grid_search_arousal.best_params_
    
    print("Tham số tốt nhất cho valence:", best_params_valence)
    print("Tham số tốt nhất cho arousal:", best_params_arousal)

    # Huấn luyện mô hình cuối cùng với tham số tốt nhất
    model_valence = RandomForestRegressor(**best_params_valence, random_state=42)
    model_valence.fit(X_train, y_valence_train)

    model_arousal = RandomForestRegressor(**best_params_arousal, random_state=42)
    model_arousal.fit(X_train, y_arousal_train)

    # Đánh giá mô hình
    y_valence_pred = model_valence.predict(X_test)
    y_arousal_pred = model_arousal.predict(X_test)
    
    print(f"Valence MSE: {mean_squared_error(y_valence_test, y_valence_pred)}")
    print(f"Arousal MSE: {mean_squared_error(y_arousal_test, y_arousal_pred)}")

    return model_valence, model_arousal, scaler

In [6]:
# Hàm dự đoán valence, arousal và chuyển thành emotion
def predict_emotion(mp3_path, model_valence, model_arousal, scaler):
    features = extract_features_from_mp3(mp3_path)
    if features is not None:
        features = scaler.transform([features])  # Chuẩn hóa đặc trưng
        valence = model_valence.predict(features)[0]
        arousal = model_arousal.predict(features)[0]
        emotion = map_to_emotion(valence, arousal)
        return emotion, valence, arousal
    else:
        return "Error", None, None

In [7]:
# Thực thi chương trình
if __name__ == "__main__":
    # Đường dẫn đến file annotation và thư mục chứa file MP3
    annotation_path = '/kaggle/input/music-detection-dataset/DEAM_Annotations/annotations/annotations averaged per song/song_level/static_annotations_averaged_songs_1_2000.csv'  # Thay bằng đường dẫn thực tế
    audio_dir = '/kaggle/input/music-detection-dataset/DEAM_audio/MEMD_audio'  # Thay bằng đường dẫn thực tế đến thư mục chứa file MP3

    # Bước 1: Chuẩn bị dữ liệu và huấn luyện mô hình
    print("Đang chuẩn bị dữ liệu huấn luyện...")
    X, y_valence, y_arousal = prepare_training_data(annotation_path, audio_dir)
    if len(X) == 0 or len(y_valence) == 0 or len(y_arousal) == 0:
        print("Không tải được dữ liệu. Vui lòng kiểm tra thư mục và file dữ liệu.")
    else:
        print("Đang huấn luyện mô hình...")
        model_valence, model_arousal, scaler = train_model(X, y_valence, y_arousal)

        joblib.dump(model_valence, 'random_forest_valence_model.pkl')
        joblib.dump(model_arousal, 'random_forest_arousal_model.pkl')
        joblib.dump(scaler, 'random_forest_scaler.pkl')

Đang chuẩn bị dữ liệu huấn luyện...
Tải các đặc trưng đã lưu trước đó...
Đã tải: 1744 mẫu
Đang huấn luyện mô hình...
Kích thước tập huấn luyện: (1395, 35), Kích thước tập kiểm tra: (349, 35)
Fitting 5 folds for each of 480 candidates, totalling 2400 fits
Fitting 5 folds for each of 480 candidates, totalling 2400 fits
Tham số tốt nhất cho valence: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 50}
Tham số tốt nhất cho arousal: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 300}
Valence MSE: 0.6475749353824511
Arousal MSE: 0.819242702404287


In [8]:
# Tải lại mô hình và scaler để sử dụng

mp3_folder_path = "/kaggle/input/music-detection-dataset/TestSong/TestSong"
output_file_path = 'music_emotion_analysis.txt'

model_valence = joblib.load('random_forest_valence_model.pkl')
model_arousal = joblib.load('random_forest_arousal_model.pkl')
scaler = joblib.load('random_forest_scaler.pkl')

if os.path.exists(mp3_folder_path) and os.path.isdir(mp3_folder_path):
    results = []  # Danh sách để lưu trữ kết quả
    print(f"Đang duyệt các file nhạc trong thư mục: {mp3_folder_path}")
    for filename in os.listdir(mp3_folder_path):
        if filename.endswith(('.mp3', '.wav', '.flac')):  # Lọc các file nhạc phổ biến
            file_path = os.path.join(mp3_folder_path, filename)
            print(f"Đang phân tích file: {filename}...")
            predicted_emotion, valence, arousal = predict_emotion(file_path, model_valence, model_arousal, scaler)
            print(f"Cảm xúc dự đoán của bài hát là: {predicted_emotion} (valence: {valence}, arousal: {arousal})")
            results.append(f"File: {filename}: {predicted_emotion} (valence: {valence}, arousal: {arousal})")

    print("Hoàn tất phân tích tất cả các file nhạc.")

    # Lưu kết quả ra file txt trong thư mục /kaggle/working/
    with open(output_file_path, 'w', encoding='utf-8') as f:
        f.write("--- Kết quả phân tích cảm xúc âm nhạc ---\n")
        for result in results:
            f.write(result + '\n')
        f.write("--- Kết thúc ---\n")

    print(f"Kết quả phân tích đã được lưu vào file: {output_file_path}")


else:
    print("Đường dẫn thư mục không hợp lệ hoặc không tồn tại. Vui lòng kiểm tra lại đường dẫn!")



Đang duyệt các file nhạc trong thư mục: /kaggle/input/music-detection-dataset/TestSong/TestSong
Đang phân tích file: su_nghiep_chuong.mp3...


<ipython-input-3-48ee95712220>:8: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  tempo = librosa.beat.tempo(y=y, sr=sr)[0]                         # 1 đặc trưng


Cảm xúc dự đoán của bài hát là: HAPPY (valence: 5.720214669077564, arousal: 5.987822936507938)
Đang phân tích file: Ngay-Chua-Giong-Bao-Bui-Lan-Huong.mp3...


<ipython-input-3-48ee95712220>:8: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  tempo = librosa.beat.tempo(y=y, sr=sr)[0]                         # 1 đặc trưng


Cảm xúc dự đoán của bài hát là: SAD (valence: 4.8366844434087914, arousal: 4.625148888888888)
Đang phân tích file: BacPhanRemix2019-JackG5RDJFuture-6058030.mp3...


...

<ipython-input-3-48ee95712220>:8: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  tempo = librosa.beat.tempo(y=y, sr=sr)[0]                         # 1 đặc trưng


Cảm xúc dự đoán của bài hát là: HAPPY (valence: 5.511919199066886, arousal: 5.633276190476195)
Đang phân tích file: BuonCuaAnh-KICMDatGMasew-9213751.mp3...


<ipython-input-3-48ee95712220>:8: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  tempo = librosa.beat.tempo(y=y, sr=sr)[0]                         # 1 đặc trưng


Cảm xúc dự đoán của bài hát là: SAD (valence: 4.941167712074309, arousal: 4.746997222222219)
Đang phân tích file: 004.mp3...
Cảm xúc dự đoán của bài hát là: HAPPY (valence: 5.238994556186697, arousal: 5.129984795574794)
Hoàn tất phân tích tất cả các file nhạc.
Kết quả phân tích đã được lưu vào file: music_emotion_analysis.txt


<ipython-input-3-48ee95712220>:8: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  tempo = librosa.beat.tempo(y=y, sr=sr)[0]                         # 1 đặc trưng
